## 3.2: Deep Learning - Tabular GAN from Scratch using PyTorch
In this section, we build a Generative Adversarial Network (GAN) layer by layer.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### Step 1: Data Preparation for Deep Learning
Neural networks only understand numbers (tensors), and they prefer them scaled between -1 and 1 or 0 and 1.

In [ ]:
# Assuming 'X_train' and 'y_train' are already in memory from the previous SMOTE step.
# We will train the GAN ONLY on the minority class (Class/ASD = 1) to generate more autistic patients.

# 1. Filter only Autistic patients
train_full = pd.concat([X_train, y_train], axis=1)
autistic_data = train_full[train_full['Class/ASD'] == 1].drop(columns=['Class/ASD']).values

# 2. Scale the data (Crucial for Neural Networks to learn stability)
scaler = StandardScaler()
autistic_scaled = scaler.fit_transform(autistic_data)

# 3. Convert to PyTorch Tensors
real_data_tensor = torch.tensor(autistic_scaled, dtype=torch.float32)

# Hyperparameters
input_dim = real_data_tensor.shape[1] # Number of features (e.g., 20)
noise_dim = 15  # The size of the random noise vector (Latent space)
batch_size = 16 # Because our dataset is small, we use a small batch size

print(f"Data ready for Neural Network. Features: {input_dim}, Noise dimension: {noise_dim}")

### Step 2: The Generator (Kẻ làm giả)
Its job is to take random noise and transform it into a fake patient record.

In [ ]:
class Generator(nn.Module):
    def __init__(self, noise_dim, output_dim):
        super(Generator, self).__init__()
        # A simple Multi-Layer Perceptron (MLP)
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 64),
            nn.ReLU(),  # Activation function: Adds non-linearity
            nn.BatchNorm1d(64), # Stabilizes training
            
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            
            nn.Linear(128, output_dim),
            # We don't use activation at the end because data is scaled via StandardScaler (can be > 1 or < -1)
        )

    def forward(self, x):
        return self.net(x)

### Step 3: The Discriminator (Cảnh sát)
Its job is to look at a patient record and predict a probability (0 to 1) of whether it is REAL or FAKE.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.2), # LeakyReLU prevents "dying neurons" in GANs
            
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            
            nn.Linear(64, 1),
            nn.Sigmoid() # Squashes the output between 0 (Fake) and 1 (Real)
        )

    def forward(self, x):
        return self.net(x)

# Instantiate the networks
generator = Generator(noise_dim, input_dim)
discriminator = Discriminator(input_dim)

print("Neural Networks initialized!")

### Step 4: The Training Loop (Cuộc chiến Min-Max)
We use Binary Cross Entropy (BCE) Loss. 
Discriminator wants BCE to be 0 (perfect classification).
Generator wants Discriminator's BCE to be high (fooled).

In [ ]:
# Optimizers (Adam is the industry standard)
lr = 0.0002
opt_gen = optim.Adam(generator.parameters(), lr=lr)
opt_disc = optim.Adam(discriminator.parameters(), lr=lr)

# Loss function
criterion = nn.BCELoss()

epochs = 500 # Number of times they fight
print("Starting the GAN Training Loop...")

for epoch in range(epochs):
    # ---------------------
    # 1. Train Discriminator
    # ---------------------
    opt_disc.zero_grad()
    
    # 1a. Train on REAL data
    # Create labels of 1s for real data
    real_labels = torch.ones(real_data_tensor.size(0), 1)
    predictions_real = discriminator(real_data_tensor)
    loss_disc_real = criterion(predictions_real, real_labels)
    
    # 1b. Train on FAKE data
    # Generate random noise
    noise = torch.randn(real_data_tensor.size(0), noise_dim)
    fake_data = generator(noise) # Forger creates fake data
    
    # Create labels of 0s for fake data. (Using .detach() so we don't calculate gradients for Generator yet)
    fake_labels = torch.zeros(real_data_tensor.size(0), 1)
    predictions_fake = discriminator(fake_data.detach())
    loss_disc_fake = criterion(predictions_fake, fake_labels)
    
    # Total Discriminator Loss
    loss_disc = loss_disc_real + loss_disc_fake
    loss_disc.backward() # Backpropagation (Calculate how wrong it was)
    opt_disc.step()      # Update weights (Police gets smarter)
    
    # ---------------------
    # 2. Train Generator
    # ---------------------
    opt_gen.zero_grad()
    
    # The Generator wants the Discriminator to think the fake data is REAL (label 1)
    predictions_fake_for_gen = discriminator(fake_data)
    loss_gen = criterion(predictions_fake_for_gen, real_labels) # Notice we use real_labels here!
    
    loss_gen.backward()
    opt_gen.step() # Update weights (Forger gets smarter)
    
    # Print progress
    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss D: {loss_disc.item():.4f} | Loss G: {loss_gen.item():.4f}")

print("Training Complete! The Generator has mastered the data distribution.")

### Step 5: Generate New Synthetic Patients

In [ ]:
# Let's generate 50 new autistic patients
num_fake_samples = 50
random_noise = torch.randn(num_fake_samples, noise_dim)

# Pass noise to the trained Generator
generator.eval() # Set to evaluation mode
with torch.no_grad():
    fake_scaled_data = generator(random_noise).numpy()

# Inverse transform to get real numbers (undo the StandardScaler)
fake_raw_data = scaler.inverse_transform(fake_scaled_data)

# Convert back to DataFrame
fake_df = pd.DataFrame(fake_raw_data, columns=X_train.columns)
# Add the Target column back (since we only trained on Class 1)
fake_df['Class/ASD'] = 1 

# Since some binary features (like A1_Score, gender) might be generated as 0.8 or 0.2, 
# we need to round them to the nearest integer.
binary_cols = ['A1_Score', 'A2_Score', 'A3_Score', 'A4_Score', 'A5_Score', 
               'A6_Score', 'A7_Score', 'A8_Score', 'A9_Score', 'A10_Score', 
               'gender', 'jaundice', 'austim', 'used_app_before']

for col in binary_cols:
    if col in fake_df.columns:
        # Clip ensures values stay between 0 and 1 before rounding
        fake_df[col] = np.clip(np.round(fake_df[col]), 0, 1).astype(int)

print(f"Successfully generated {num_fake_samples} highly realistic synthetic autistic patients!")
display(fake_df.head())